In [1]:
import numpy as np
import pandas as pd

In [2]:
np.random.seed(0)
muestras = 100

# Generar 100 instancias para X1, X2, X3 y X5
X1 = np.random.randint(0, 2, muestras)
X2 = np.random.randint(0, 2, muestras)
X3 = np.random.randint(0, 2, muestras)
X5 = np.random.randint(0, 2, muestras)

# Calcular Y como XOR de X1, X2, X3
Y = X1 ^ X2 ^ X3

# Calcular X4 como XOR de X2, X3
X4 = X2 ^ X3

# Crear el DataFrame
data = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'X5': X5, 'Y': Y})

# Guardar los datos en un archivo CSV
# data.to_csv('synthetic_dataset.csv', index=False)

data.head()

,X1,X2,X3,X4,X5,Y
0,0,1,1,0,1,0
1,1,0,0,0,0,1
2,1,0,1,1,1,0
3,0,1,0,1,0,1
4,1,0,0,0,0,1


# Tecnicas

## Información mutua

In [3]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score

# Separar variables predictoras y objetivo
X = data.drop(columns=['Y'])  
y = data['Y']

# Calcular información mutua
mi_scores = mutual_info_classif(X, y, discrete_features=True)

# Crear un DataFrame con los resultados
mi_results = pd.DataFrame({'Feature': X.columns, 'Mutual Information': mi_scores})
mi_results = mi_results.sort_values(by='Mutual Information', ascending=False)

# Mostrar los resultados
print("Ranking por Información Mutua:")
print(mi_results)


Ranking por Información Mutua:
  Feature  Mutual Information
3      X4            0.006485
1      X2            0.000694
2      X3            0.000123
0      X1            0.000067
4      X5            0.000035


## Chi2

In [4]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

X = data.drop(columns=['Y'])  
y = data['Y']

# Normalizar los datos a valores positivos para el test chi2
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar test chi2
chi2_scores, p_values = chi2(X_scaled, y)

# Crear un DataFrame con los resultados
chi2_results = pd.DataFrame({'Feature': X.columns, 'Chi2 Score': chi2_scores, 'p-value': p_values})
chi2_results = chi2_results.sort_values(by='Chi2 Score', ascending=False)

# Mostrar los resultados
print("Ranking por Test de Chi-Cuadrado:")
print(chi2_results)


Ranking por Test de Chi-Cuadrado:
  Feature  Chi2 Score   p-value
3      X4    0.621784  0.430385
1      X2    0.076401  0.782236
2      X3    0.012018  0.912707
0      X1    0.005942  0.938555
4      X5    0.002830  0.957577


## Relief

In [5]:
from skrebate import ReliefF

X = data.drop(columns=['Y'])  
y = data['Y']

# Aplicar ReliefF
relief = ReliefF(n_neighbors=10)  # Número de vecinos a considerar
relief.fit(X.values, y.values)

# Crear un DataFrame con los resultados
relief_results = pd.DataFrame({'Feature': X.columns, 'Relief Score': relief.feature_importances_})
relief_results = relief_results.sort_values(by='Relief Score', ascending=False)

# Mostrar los resultados
print("Ranking por Relief:")
print(relief_results)


Ranking por Relief:
  Feature  Relief Score
0      X1         0.688
3      X4         0.306
2      X3         0.123
1      X2         0.085
4      X5        -0.217


## PCA

In [6]:
from sklearn.decomposition import PCA

X = data.drop(columns=['Y'])  
y = data['Y']

# Aplicar PCA
pca = PCA(n_components=5)  # Mantener el número total de características
pca.fit(X)

# Crear un DataFrame con la varianza explicada por cada componente
pca_results = pd.DataFrame({'Feature': X.columns, 'Explained Variance': pca.explained_variance_ratio_})
pca_results = pca_results.sort_values(by='Explained Variance', ascending=False)

# Mostrar los resultados
print("Ranking por PCA:")
print(pca_results)


Ranking por PCA:
  Feature  Explained Variance
0      X1            0.272070
1      X2            0.211571
2      X3            0.190997
3      X4            0.182295
4      X5            0.143067


## RFE-SVM

In [7]:
from sklearn.feature_selection import RFE
from sklearn.svm import SVC

X = data.drop(columns=['Y'])  
y = data['Y']

# Definir el modelo SVM
svm = SVC(kernel="linear")

# Aplicar RFE
rfe = RFE(estimator=svm, n_features_to_select=1)  # Para obtener el ranking completo
rfe.fit(X, y)

# Crear un DataFrame con los rankings
rfe_results = pd.DataFrame({'Feature': X.columns, 'RFE Ranking': rfe.ranking_})
rfe_results = rfe_results.sort_values(by='RFE Ranking')

# Mostrar los resultados
print("Ranking por RFE-SVM:")
print(rfe_results)


Ranking por RFE-SVM:
  Feature  RFE Ranking
3      X4            1
0      X1            2
2      X3            3
1      X2            4
4      X5            5
